# 02 - RAC Evaluation (With Alpha Sweep)

## 1. Notebook Goal
Benchmark baseline and RAC scoring variants on the fixed test split while tuning alpha for density-aware methods.

## 2. Experimental Design
- Load test records and retrieval collections from the populated database.
- Evaluate the traditional baseline under the same protocol.
- Evaluate RAC variants and sweep alpha for local/KDE DNDS.
- Run secondary sensitivity checks (k-density and KDE bandwidth).
- Save metrics, predictions, and summary figures.

## 3. Inputs
- dataset/CVPR_2024_dataset_Test/
- dataset_text/test.csv
- chroma_db/ populated image/text collections
- models/ and text_models/ checkpoints

## 4. Outputs
- results/phase2/evaluation_results.json
- results/phase2/metrics_summary.csv
- results/phase2/scoring_variants_predictions.csv
- figures/phase2/ evaluation plots

## 5. Execution Guide
Run all cells in order. This notebook is the tuned reference used by the final Phase 2 summary.

### Cell 2 - Imports, Paths, and Runtime Controls
Purpose: initialize dependencies, hardware/runtime options, model/checkpoint paths, logging settings, and output file locations for the evaluation pipeline.

In [1]:
import importlib
import json
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

from src.phase2.config import get_phase2_config
from src.phase2.data_utils import build_records_from_csv
from src.phase2.db_client import (
    get_class_counts,
    get_image_collection,
    get_persistent_client,
    get_text_collection,
)
import src.phase2.evaluation as phase2_evaluation

importlib.reload(phase2_evaluation)
from src.phase2.evaluation import (
    compute_metrics,
    evaluate_variant,
    save_alpha_sweep_csv,
    save_metrics_summary_csv,
    save_predictions_csv,
    save_results,
)
from src.phase2.gpu_utils import (
    get_device,
    print_device_info,
    print_gpu_memory,
)
from src.phase2.imbalance import infer_class_groups
from src.phase2.scoring import (
    global_dnds,
    idw,
    kde_dnds,
    local_dnds,
    majority_vote,
)
from src.phase2.visualization import (
    plot_alpha_sensitivity,
    plot_confusion_matrices,
    plot_kde_bandwidth_ablation,
    plot_phase2_vs_phase1,
    plot_scoring_comparison,
)

CONFIG = get_phase2_config()

# Per-notebook GPU controls
PREFER_GPU = True
CLEANUP_INTERVAL = 0
MEMORY_LOG_INTERVAL = 0

# Hyperparameter sweep controls
PARALLEL_SWEEPS = True
SWEEP_MAX_WORKERS = 20
SWEEP_LOG_INTERVAL = 250

DEVICE = get_device(prefer_gpu=PREFER_GPU)

REPO_ROOT = Path("../..").resolve()
VAL_DIR = REPO_ROOT / "dataset" / "CVPR_2024_dataset_Val"
VAL_CSV = REPO_ROOT / "dataset_text" / "val.csv"
TEST_DIR = REPO_ROOT / "dataset" / "CVPR_2024_dataset_Test"
TEST_CSV = REPO_ROOT / "dataset_text" / "test.csv"
PHASE1_RESULTS_PATH = REPO_ROOT / "results" / "multimodal_fusion_results.json"
RESULTS_PATH = REPO_ROOT / "results" / "phase2" / "evaluation_results.json"
METRICS_CSV_PATH = REPO_ROOT / "results" / "phase2" / "metrics_summary.csv"
ALPHA_CSV_PATH = REPO_ROOT / "results" / "phase2" / "alpha_sweep.csv"
PREDICTIONS_CSV_PATH = REPO_ROOT / "results" / "phase2" / "scoring_variants_predictions.csv"
SWEEP_LOG_PATH = REPO_ROOT / "results" / "phase2" / "evaluation_sweeps.log"
FIGURES_DIR = REPO_ROOT / "figures" / "phase2"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not VAL_DIR.exists() or not VAL_CSV.exists():
    raise FileNotFoundError(
        "Validation dataset missing. Expected dataset and dataset_text paths in repo root."
    )

if not TEST_DIR.exists() or not TEST_CSV.exists():
    raise FileNotFoundError(
        "Test dataset missing. Expected dataset and dataset_text paths in repo root."
    )

if not PHASE1_RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Phase 1 fusion JSON not found at {PHASE1_RESULTS_PATH}"
    )

if PARALLEL_SWEEPS:
    SWEEP_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    SWEEP_LOG_PATH.write_text("", encoding="utf-8")

print_device_info(DEVICE)
if MEMORY_LOG_INTERVAL > 0:
    print_gpu_memory(prefix="Startup GPU memory", device=DEVICE)

if PARALLEL_SWEEPS:
    print(f"Sweep progress log: {SWEEP_LOG_PATH}")

d:\University of Calgary\Winter 2026\ENSF617-Introduction-To-Machine-Learning\trash-classification-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU (4.00 GB VRAM, allocated 0.00 GB, reserved 0.00 GB)
Sweep progress log: D:\University of Calgary\Winter 2026\ENSF617-Introduction-To-Machine-Learning\trash-classification-project\results\phase2\evaluation_sweeps.log


### Cell 3 - Load Validation/Test Data and Collections
Purpose: construct validation/test samples from CSV/images, connect to ChromaDB collections, infer majority/minority class groups, and register scoring variants for evaluation.

In [2]:
val_samples, val_missing_examples, val_total_rows = build_records_from_csv(
    csv_path=VAL_CSV,
    split_dir=VAL_DIR,
    text_column="text",
    label_column="label",
    text_key="text",
)

test_samples, test_missing_examples, test_total_rows = build_records_from_csv(
    csv_path=TEST_CSV,
    split_dir=TEST_DIR,
    text_column="text",
    label_column="label",
    text_key="text",
)

if val_missing_examples:
    print("Skipped validation rows with missing image files (up to 10 shown):")
    for item in val_missing_examples:
        print(f"  - {item}")

if test_missing_examples:
    print("Skipped test rows with missing image files (up to 10 shown):")
    for item in test_missing_examples:
        print(f"  - {item}")

if not val_samples:
    raise RuntimeError("No validation samples found after image path resolution.")
if not test_samples:
    raise RuntimeError("No test samples found after image path resolution.")

print(f"Validation samples available for tuning: {len(val_samples)} / {val_total_rows}")
print(f"Test samples available for final evaluation: {len(test_samples)} / {test_total_rows}")

client = get_persistent_client(str(REPO_ROOT / "chroma_db"))
image_collection = get_image_collection(client)
text_collection = get_text_collection(client)

image_class_counts = get_class_counts(image_collection)
majority_classes, minority_classes = infer_class_groups(
    image_class_counts, threshold=float(CONFIG["majority_threshold"])
)
CONFIG["majority_classes"] = majority_classes
CONFIG["minority_classes"] = minority_classes
print(
    f"Dynamic class grouping -> majority: {majority_classes}, minority: {minority_classes}"
)

variant_fns = {
    "majority_vote": majority_vote,
    "idw": idw,
    "global_dnds": global_dnds,
    "local_dnds": local_dnds,
    "kde_dnds": kde_dnds,
}

Validation samples available for tuning: 1820 / 1820
Test samples available for final evaluation: 3452 / 3452


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 756.43it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dynamic class grouping -> majority: ['Blue'], minority: ['Black', 'Green', 'TTR']


### Cell 4 - RAC Tuning (Validation) and Final RAC Metrics (Test)
Purpose: tune RAC hyperparameters on validation samples, then evaluate the tuned RAC variants on the held-out test set.

In [ ]:
results = {
    "variants": {},
    "alpha_sweep": {
        "local_dnds": {"alphas": [], "macro_f1": []},
        "kde_dnds": {"alphas": [], "macro_f1": []},
    },
    "kde_bandwidth": {"bandwidths": [], "macro_f1": []},
    "k_density_sweep": {"values": [], "macro_f1": []},
}


def _evaluate_on_samples(
    variant_fn,
    sample_records,
    *,
    alpha=None,
    config_overrides=None,
    score_kwargs=None,
    run_label=None,
):
    eval_config = dict(CONFIG)
    if config_overrides:
        eval_config.update(config_overrides)
    return evaluate_variant(
        variant_fn,
        sample_records,
        image_collection,
        text_collection,
        eval_config,
        alpha=alpha,
        score_kwargs=score_kwargs,
        cleanup_interval=CLEANUP_INTERVAL,
        memory_log_interval=MEMORY_LOG_INTERVAL,
        show_progress=not PARALLEL_SWEEPS,
        run_label=run_label,
        progress_log_path=str(SWEEP_LOG_PATH) if PARALLEL_SWEEPS else None,
        progress_log_interval=SWEEP_LOG_INTERVAL,
    )


def _execute_jobs(jobs):
    if PARALLEL_SWEEPS and len(jobs) > 1:
        workers = max(1, min(SWEEP_MAX_WORKERS, len(jobs)))
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = [pool.submit(job) for job in jobs]
            return [future.result() for future in futures]
    return [job() for job in jobs]


# Evaluate non-tuned variants directly on test split.
for name, fn in variant_fns.items():
    if name in {"local_dnds", "kde_dnds"}:
        continue
    metrics = _evaluate_on_samples(
        fn,
        test_samples,
        run_label=f"test::variant::{name}",
    )
    results["variants"][name] = metrics

# Alpha sweeps for local_dnds and kde_dnds on validation split.
alpha_jobs = []
for alpha in CONFIG["alpha_sweep"]:
    alpha_value = float(alpha)

    def _local_alpha_job(a=alpha_value):
        metrics = _evaluate_on_samples(
            local_dnds,
            val_samples,
            alpha=a,
            run_label=f"val::sweep::local_dnds::alpha={a}",
        )
        return "local_dnds", a, metrics["macro_f1"]

    def _kde_alpha_job(a=alpha_value):
        metrics = _evaluate_on_samples(
            kde_dnds,
            val_samples,
            alpha=a,
            run_label=f"val::sweep::kde_dnds::alpha={a}",
        )
        return "kde_dnds", a, metrics["macro_f1"]

    alpha_jobs.extend([_local_alpha_job, _kde_alpha_job])

alpha_points = _execute_jobs(alpha_jobs)
local_points = sorted(
    [(a, score) for name, a, score in alpha_points if name == "local_dnds"],
    key=lambda item: item[0],
)
kde_points = sorted(
    [(a, score) for name, a, score in alpha_points if name == "kde_dnds"],
    key=lambda item: item[0],
)
results["alpha_sweep"]["local_dnds"]["alphas"] = [a for a, _ in local_points]
results["alpha_sweep"]["local_dnds"]["macro_f1"] = [score for _, score in local_points]
results["alpha_sweep"]["kde_dnds"]["alphas"] = [a for a, _ in kde_points]
results["alpha_sweep"]["kde_dnds"]["macro_f1"] = [score for _, score in kde_points]

best_alpha_local = max(local_points, key=lambda item: item[1])[0]
best_alpha_kde = max(kde_points, key=lambda item: item[1])[0]
results["best_alpha_local"] = best_alpha_local
results["best_alpha_kde"] = best_alpha_kde

# Local DNDS K_density sweep on validation split (using best local alpha).
k_density_jobs = []
for k_value in CONFIG.get("K_density_sweep", [CONFIG["K_density"]]):
    k_density = int(k_value)

    def _k_density_job(k=k_density):
        metrics = _evaluate_on_samples(
            local_dnds,
            val_samples,
            alpha=best_alpha_local,
            config_overrides={"K_density": k},
            run_label=f"val::sweep::local_dnds::K_density={k}",
        )
        return k, metrics["macro_f1"]

    k_density_jobs.append(_k_density_job)

k_density_points = sorted(_execute_jobs(k_density_jobs), key=lambda item: item[0])
results["k_density_sweep"]["values"] = [k for k, _ in k_density_points]
results["k_density_sweep"]["macro_f1"] = [score for _, score in k_density_points]
best_k_density = max(k_density_points, key=lambda item: item[1])[0]
results["best_k_density_local"] = int(best_k_density)
CONFIG["K_density"] = int(best_k_density)

# Final local_dnds evaluation on test split.
results["variants"]["local_dnds"] = _evaluate_on_samples(
    local_dnds,
    test_samples,
    alpha=best_alpha_local,
    config_overrides={"K_density": int(best_k_density)},
    run_label=f"test::final::local_dnds::alpha={best_alpha_local}::K_density={best_k_density}",
)

# KDE bandwidth ablation on validation split.
kde_bandwidth_jobs = []
for bandwidth in CONFIG["kde_bandwidth_sweep"]:
    bw = float(bandwidth)

    def _kde_bandwidth_job(b=bw):
        metrics = _evaluate_on_samples(
            kde_dnds,
            val_samples,
            alpha=best_alpha_kde,
            score_kwargs={"bandwidth": b},
            run_label=f"val::sweep::kde_dnds::bandwidth={b}",
        )
        return b, metrics["macro_f1"]

    kde_bandwidth_jobs.append(_kde_bandwidth_job)

kde_bandwidth_points = sorted(
    _execute_jobs(kde_bandwidth_jobs), key=lambda item: item[0]
)
results["kde_bandwidth"]["bandwidths"] = [b for b, _ in kde_bandwidth_points]
results["kde_bandwidth"]["macro_f1"] = [score for _, score in kde_bandwidth_points]
best_bw_idx = max(
    range(len(results["kde_bandwidth"]["macro_f1"])),
    key=lambda i: results["kde_bandwidth"]["macro_f1"][i],
)
results["best_kde_bandwidth"] = results["kde_bandwidth"]["bandwidths"][best_bw_idx]
CONFIG["kde_bandwidth"] = float(results["best_kde_bandwidth"])

# Final kde_dnds evaluation on test split.
results["variants"]["kde_dnds"] = _evaluate_on_samples(
    kde_dnds,
    test_samples,
    alpha=best_alpha_kde,
    score_kwargs={"bandwidth": float(results["best_kde_bandwidth"])},
    run_label=(
        f"test::final::kde_dnds::alpha={best_alpha_kde}::bandwidth={results['best_kde_bandwidth']}"
    ),
)

print(f"Best local DNDS alpha from validation sweep: {best_alpha_local}")
print(f"Best local DNDS K_density from validation sweep: {best_k_density}")
print(f"Best KDE-DNDS alpha from validation sweep: {best_alpha_kde}")
print(f"Best KDE bandwidth from validation sweep: {results['best_kde_bandwidth']}")
print("Final RAC variant metrics are reported on the test split.")
if PARALLEL_SWEEPS:
    print(f"Sweep progress logged to: {SWEEP_LOG_PATH}")

### Cell 5 - RAC Variants with Alpha Sweep
Purpose: execute RAC scoring variants and sweep alpha values for density-aware methods to select the best operating point under the same evaluation protocol.

In [ ]:
# Load traditional baseline metrics from Phase 1 multimodal JSON export.
with PHASE1_RESULTS_PATH.open("r", encoding="utf-8") as f:
    phase1_payload = json.load(f)

phase1_meta = phase1_payload.get("metadata", {})
phase1_final = phase1_payload.get("final_test_results", {})
phase1_fused = phase1_final.get("fused", {})
index_to_label = phase1_meta.get("class_names", CONFIG["class_names"])
label_indices = phase1_final.get("labels", [])
pred_indices = phase1_fused.get("predictions", [])

if not label_indices or not pred_indices:
    raise ValueError(
        "Phase 1 JSON is missing labels/predictions required for traditional baseline metrics."
    )
if len(label_indices) != len(pred_indices):
    raise ValueError("Phase 1 labels and fused predictions have different lengths.")

y_true_labels = [index_to_label[int(i)] for i in label_indices]
y_pred_labels = [index_to_label[int(i)] for i in pred_indices]

traditional_metrics = compute_metrics(y_true_labels, y_pred_labels, CONFIG["class_names"])
traditional_latency = float(phase1_fused.get("inference_time_ms", 0.0))
traditional_metrics["inference_time_ms"] = traditional_latency
traditional_metrics["latency_ms_per_sample"] = traditional_latency
traditional_metrics["y_true"] = y_true_labels
traditional_metrics["y_pred"] = y_pred_labels
traditional_metrics["alpha"] = float(phase1_meta.get("selected_alpha", CONFIG["alpha"]))
traditional_metrics["variant"] = "traditional"
traditional_metrics["source"] = str(PHASE1_RESULTS_PATH)

results["variants"]["traditional"] = traditional_metrics
print(f"Loaded traditional baseline metrics from: {PHASE1_RESULTS_PATH}")

### Cell 6 - Secondary Ablations (k-Density and KDE Bandwidth)
Purpose: run targeted ablations that vary k-density and KDE bandwidth to analyze sensitivity beyond alpha and capture stable default settings.

In [ ]:
best_alpha_local_idx = max(
    range(len(results["alpha_sweep"]["local_dnds"]["macro_f1"])),
    key=lambda i: results["alpha_sweep"]["local_dnds"]["macro_f1"][i],
)
best_alpha_kde_idx = max(
    range(len(results["alpha_sweep"]["kde_dnds"]["macro_f1"])),
    key=lambda i: results["alpha_sweep"]["kde_dnds"]["macro_f1"][i],
)
results["best_alpha_local"] = results["alpha_sweep"]["local_dnds"]["alphas"][
    best_alpha_local_idx
]
results["best_alpha_kde"] = results["alpha_sweep"]["kde_dnds"]["alphas"][
    best_alpha_kde_idx
]

save_results(results, str(RESULTS_PATH))
save_metrics_summary_csv(results["variants"], str(METRICS_CSV_PATH))
save_alpha_sweep_csv(results["alpha_sweep"]["local_dnds"], str(ALPHA_CSV_PATH))
save_predictions_csv(results["variants"], str(PREDICTIONS_CSV_PATH))

plot_scoring_comparison(results, str(FIGURES_DIR / "scoring_comparison.png"))
plot_alpha_sensitivity(
    results["alpha_sweep"], str(FIGURES_DIR / "alpha_sensitivity.png")
)
plot_kde_bandwidth_ablation(
    results["kde_bandwidth"], str(FIGURES_DIR / "kde_bandwidth_ablation.png")
)
plot_confusion_matrices(
    results,
    CONFIG["class_names"],
    str(FIGURES_DIR / "confusion_matrices_phase2.png"),
    variants=[
        "majority_vote",
        "idw",
        "global_dnds",
        "local_dnds",
        "kde_dnds",
        "traditional",
    ],
)
phase1_macro_f1 = results["variants"]["traditional"].get("macro_f1", 0.0)
best_phase2 = max(
    v.get("macro_f1", 0.0) for k, v in results["variants"].items() if k != "traditional"
)
plot_phase2_vs_phase1(
    best_phase2, phase1_macro_f1, str(FIGURES_DIR / "phase2_vs_phase1.png")
)

print(f"Saved JSON results: {RESULTS_PATH}")
print(f"Saved CSV metrics: {METRICS_CSV_PATH}")
print(f"Saved CSV alpha sweep (local_dnds on validation): {ALPHA_CSV_PATH}")
print(f"Saved CSV predictions: {PREDICTIONS_CSV_PATH}")

### Cell 7 - Save Artifacts and Generate Figures
Purpose: persist full metrics/predictions to disk and render summary plots (variant comparison, alpha behavior, confusion matrices, and phase comparisons).

### Cell 8 - Final Printed Summary
Purpose: print a compact, human-readable summary table of key metrics and selected hyperparameters for quick comparison in the notebook output.

In [ ]:
print("=" * 60)
print("PHASE 2 RAC EXPERIMENT - RESULTS SUMMARY")
print("=" * 60)
print("Variant              | Accuracy | Macro F1 | TTR F1 | Latency")
print("---------------------|----------|----------|--------|--------")
for name, metrics in results["variants"].items():
    ttr_f1 = metrics.get("per_class_f1", {}).get("TTR", 0.0)
    print(
        f"{name:<20} | {metrics['accuracy']*100:6.2f}% | {metrics['macro_f1']:.4f} | {ttr_f1:.4f} | {metrics['latency_ms_per_sample']:.2f} ms"
    )
print("=" * 60)
print(f"Best alpha (local DNDS): {results['best_alpha_local']}")
print(f"Best alpha (KDE-DNDS): {results['best_alpha_kde']}")
print(f"Best KDE bandwidth (h): {results['best_kde_bandwidth']}")